# 02 - El pipeline en acción, etapa por etapa

Este notebook toma contratos reales y muestra cómo se transforma el texto en cada etapa del
pipeline de SECOPEDIA. Sus salidas son las que se citan en el documento final.

Requisitos: haber ejecutado `descargar_datos.py`, `preparar_datos.py`, `construir_lugares.py`,
`preprocesar.py` y entrenado los modelos con `entrenar_linea_base.py`, `entrenar_red_neuronal.py` y
`entrenar_beto.py`.

In [1]:
import json

import joblib
import pandas as pd

from secopedia.configuracion import RAIZ_PROYECTO, cargar_config, ruta
from secopedia.etiquetas import extraer_segmento
from secopedia.limpieza import clave_texto
from secopedia.normalizacion import Normalizador
from secopedia.prediccion import Predictor
from secopedia.representacion import terminos_principales
from secopedia.segmentos import nombre_segmento
from secopedia.tokenizacion import cargar_spacy, lematizar

pd.set_option("display.max_colwidth", 200)
config = cargar_config()
normalizador = Normalizador.desde_archivo(ruta(config["rutas"]["lugares"]))
nlp = cargar_spacy()
modelo = joblib.load(RAIZ_PROYECTO / config["rutas"]["modelos"] / "regresion_logistica.joblib")
vectorizador = modelo.named_steps["tfidf"]

## Contratos de ejemplo

Se usan contratos de la tabla de muestra de la Entrega 1 (texto y código tal como están en SECOP II).

In [2]:
ejemplos = pd.DataFrame([
    ("COMPRA DE PRODUCTOS DE CAFETERIA PARA LA ENTIDAD MUNICIPIO DE YUMBO VIGENCIA 2023", "V1.50201700"),
    ("ADQUISICION DE LA POLIZA DE MANEJO GLOBAL PARA LA PERSONERIA MUNICIPAL DE SOTAQUIRA BOYACA", "V1.84131500"),
    ("Contratar el suministro de combustible; lubricantes; llantas y baterías para los vehículos y el aula móvil del Centro Agroturístico Regional Santander", "V1.15101506"),
    ("Adquisición de implementos; insumos y medicamentos para prestar los servicios de medicina veterinaria en el municipio de Caldas; Antioquia", "V1.42121500"),
    ("Adquisición de mobiliario para las sedes de la Delegación Departamental de la Registraduría Nacional del Estado Civil en el departamento de Magdalena.", "V1.72153606"),
], columns=["objeto_del_contrato", "codigo_de_categoria_principal"])
ejemplos

,objeto_del_contrato,codigo_de_categoria_principal
0,COMPRA DE PRODUCTOS DE CAFETERIA PARA LA ENTIDAD MUNICIPIO DE YUMBO VIGENCIA 2023,V1.50201700
1,ADQUISICION DE LA POLIZA DE MANEJO GLOBAL PARA LA PERSONERIA MUNICIPAL DE SOTAQUIRA BOYACA,V1.84131500
2,Contratar el suministro de combustible; lubricantes; llantas y baterías para los vehículos y el aula móvil del Centro Agroturístico Regional Santander,V1.15101506
3,Adquisición de implementos; insumos y medicamentos para prestar los servicios de medicina veterinaria en el municipio de Caldas; Antioquia,V1.42121500
4,Adquisición de mobiliario para las sedes de la Delegación Departamental de la Registraduría Nacional del Estado Civil en el departamento de Magdalena.,V1.72153606


## Etapas 2 y 3: clave de deduplicación y etiqueta

La clave de texto (minúsculas, sin tildes ni signos) detecta duplicados. La etiqueta es el
segmento: los 2 primeros dígitos del código UNSPSC.

In [3]:
clases = json.loads((ruta(config["rutas"]["metricas"]) / "clases.json").read_text(encoding="utf-8"))
etapa_2_3 = pd.DataFrame({
    "clave_texto": ejemplos["objeto_del_contrato"].map(clave_texto),
    "codigo": ejemplos["codigo_de_categoria_principal"],
    "segmento": ejemplos["codigo_de_categoria_principal"].map(extraer_segmento),
})
etapa_2_3["etiqueta"] = etapa_2_3["segmento"].where(etapa_2_3["segmento"].isin(clases), "otros")
etapa_2_3["nombre"] = etapa_2_3["etiqueta"].map(nombre_segmento)
etapa_2_3

,clave_texto,codigo,segmento,etiqueta,nombre
0,compra de productos de cafeteria para la entidad municipio de yumbo vigencia 2023,V1.50201700,50,50,"Alimentos, bebidas y tabaco"
1,adquisicion de la poliza de manejo global para la personeria municipal de sotaquira boyaca,V1.84131500,84,84,Servicios financieros y de seguros
2,contratar el suministro de combustible lubricantes llantas y baterias para los vehiculos y el aula movil del centro agroturistico regional santander,V1.15101506,15,15,"Combustibles, aditivos para combustibles, lubricantes y anticorrosivos"
3,adquisicion de implementos insumos y medicamentos para prestar los servicios de medicina veterinaria en el municipio de caldas antioquia,V1.42121500,42,42,"Equipo médico, accesorios y suministros"
4,adquisicion de mobiliario para las sedes de la delegacion departamental de la registraduria nacional del estado civil en el departamento de magdalena,V1.72153606,72,72,"Servicios de edificación, construcción y mantenimiento"


## Etapa 4: normalización

Minúsculas, sin tildes, separadores a espacios, años a `<AÑO>`, números a `<NUM>` y
municipios o departamentos a `<LUGAR>`.

In [4]:
ejemplos["normalizado"] = ejemplos["objeto_del_contrato"].map(normalizador)
for original, normalizado in zip(ejemplos["objeto_del_contrato"], ejemplos["normalizado"]):
    print(f"ORIGINAL:    {original}\nNORMALIZADO: {normalizado}\n")

ORIGINAL:    COMPRA DE PRODUCTOS DE CAFETERIA PARA LA ENTIDAD MUNICIPIO DE YUMBO VIGENCIA 2023
NORMALIZADO: compra de productos de cafeteria para la entidad municipio de <LUGAR> vigencia <AÑO>

ORIGINAL:    ADQUISICION DE LA POLIZA DE MANEJO GLOBAL PARA LA PERSONERIA MUNICIPAL DE SOTAQUIRA BOYACA
NORMALIZADO: adquisicion de la poliza de manejo global para la personeria municipal de <LUGAR>

ORIGINAL:    Contratar el suministro de combustible; lubricantes; llantas y baterías para los vehículos y el aula móvil del Centro Agroturístico Regional Santander
NORMALIZADO: contratar el suministro de combustible lubricantes llantas y baterias para los vehiculos y el aula movil del centro agroturistico regional <LUGAR>

ORIGINAL:    Adquisición de implementos; insumos y medicamentos para prestar los servicios de medicina veterinaria en el municipio de Caldas; Antioquia
NORMALIZADO: adquisicion de implementos insumos y medicamentos para prestar los servicios de medicina veterinaria en el municipio

## Etapa 5: tokenización, stopwords y lematización

spaCy (`es_core_news_sm`) separa los tokens, quita stopwords del español y del dominio
(por ejemplo "adquisicion", "contratar", "municipio") y deja el lema de cada palabra.

In [5]:
ejemplos["lemas"] = list(lematizar(ejemplos["normalizado"], nlp))
doc = nlp(ejemplos["normalizado"].iat[0])
print("Tokens de spaCy del primer ejemplo:")
print(pd.DataFrame([(t.text, t.lemma_, t.is_stop) for t in doc], columns=["token", "lema", "stopword"]).T.to_string(header=False))
print()
for normalizado, lemas in zip(ejemplos["normalizado"], ejemplos["lemas"]):
    print(f"{normalizado}\n  -> {lemas}\n")

Tokens de spaCy del primer ejemplo:
token     compra    de  productos    de  cafeteria  para    la  entidad  municipio    de  <LUGAR>  vigencia  <AÑO>
lema      compra    de   producto    de  cafeteria  para    el  entidad  municipio    de  <lugar>  vigencia  <año>
stopword   False  True      False  True      False  True  True    False      False  True    False     False  False

compra de productos de cafeteria para la entidad municipio de <LUGAR> vigencia <AÑO>
  -> ['producto', 'cafeteria']

adquisicion de la poliza de manejo global para la personeria municipal de <LUGAR>
  -> ['poliza', 'manejo', 'global', 'personeria']

contratar el suministro de combustible lubricantes llantas y baterias para los vehiculos y el aula movil del centro agroturistico regional <LUGAR>
  -> ['suministro', 'combustible', 'lubricant', 'llanta', 'bateria', 'vehiculo', 'aula', 'movil', 'centro', 'agroturistico', 'regional']

adquisicion de implementos insumos y medicamentos para prestar los servicios de med

## Etapa 6a: representación TF-IDF (regresión logística y SVM)

Cada contrato se convierte en un vector disperso de unigramas y bigramas de lemas. Se muestran
los términos con más peso en cada ejemplo.

In [6]:
print(f"Tamaño del vocabulario TF-IDF: {len(vectorizador.vocabulary_):,} términos")
for lemas in ejemplos["lemas"]:
    texto = " ".join(lemas)
    print(f"{texto}\n  -> {terminos_principales(vectorizador, texto)}\n")

Tamaño del vocabulario TF-IDF: 85,756 términos
producto cafeteria
  -> [('producto cafeteria', 0.709), ('cafeteria', 0.556), ('producto', 0.434)]

poliza manejo global personeria
  -> [('global personeria', 0.573), ('poliza manejo', 0.398), ('manejo global', 0.371), ('global', 0.354), ('personeria', 0.327)]

suministro combustible lubricant llanta bateria vehiculo aula movil centro agroturistico regional
  -> [('vehiculo aula', 0.313), ('lubricant llanta', 0.295), ('agroturistico regional', 0.291), ('bateria vehiculo', 0.291), ('centro agroturistico', 0.278)]

implemento insumos medicamento medicina veterinario
  -> [('medicina veterinario', 0.469), ('implemento insumos', 0.457), ('insumos medicamento', 0.451), ('veterinario', 0.311), ('medicina', 0.285)]

mobiliario sed delegacion registraduria nacional civil
  -> [('mobiliario sed', 0.44), ('delegacion registraduria', 0.426), ('nacional civil', 0.349), ('registraduria nacional', 0.349), ('delegacion', 0.308)]



## Etapa 6b: embeddings fastText y entrada de la CNN

fastText representa cada lema con 100 números; las palabras de uso parecido, o mal escritas, quedan cerca.
La CNN recibe la secuencia de índices de los lemas, rellenada hasta 40 posiciones.

In [7]:
from gensim.models import FastText

from secopedia.modelos.red_neuronal import ClasificadorCNN, codificar

carpeta_red = RAIZ_PROYECTO / config["rutas"]["modelos"] / "red_neuronal"
fasttext = FastText.load(str(carpeta_red / "fasttext.model"))
for palabra in ["combustible", "poliza", "refrigerio"]:
    print(f"{palabra}: {[vecino for vecino, _ in fasttext.wv.most_similar(palabra, topn=5)]}")

cnn = ClasificadorCNN.cargar(carpeta_red / "cnn.pt")
indices = codificar(ejemplos["lemas"].tolist(), cnn.vocabulario)
print(f"\nEntrada de la CNN: {indices.shape}")
print(f"{ejemplos['lemas'].iat[1]}\n  -> {indices[1][:8]} ... (0 es relleno, 1 es palabra desconocida)")

combustible: ['combustibles', 'combustibl', 'combustirble', 'acpm', 'gasolina']
poliza: ['polizo', 'polizar', 'amparar', 'seguro', 'ampar']
refrigerio: ['refigerio', 'almuerzo', 'regrigerio', 'hidratacion', 'cena']

Entrada de la CNN: (5, 40)
['poliza', 'manejo', 'global', 'personeria']
  -> [14585 11547  8793 14154     0     0     0     0] ... (0 es relleno, 1 es palabra desconocida)


## Etapa 6c: subpalabras de BETO

BETO recibe el texto **normalizado** (etapa 4), no los lemas, y lo parte en subpalabras de su vocabulario.
`##` indica que la subpalabra va pegada a la anterior.

In [8]:
from secopedia.modelos.beto import ClasificadorBeto

carpeta_beto = RAIZ_PROYECTO / config["rutas"]["modelos"] / "beto_secopedia"
beto = ClasificadorBeto(str(carpeta_beto))
for texto in ejemplos["normalizado"].head(2):
    print(f"{texto}\n  -> {beto.tokenizador.tokenize(texto)}\n")

compra de productos de cafeteria para la entidad municipio de <LUGAR> vigencia <AÑO>
  -> ['compra', 'de', 'productos', 'de', 'cafe', '##teria', 'para', 'la', 'entidad', 'municipio', 'de', '<', 'lugar', '>', 'vigencia', '<', 'año', '>']

adquisicion de la poliza de manejo global para la personeria municipal de <LUGAR>
  -> ['adquis', '##icion', 'de', 'la', 'poli', '##za', 'de', 'manejo', 'global', 'para', 'la', 'person', '##eria', 'municipal', 'de', '<', 'lugar', '>']



## Etapa 7: predicción de la regresión logística (top 3)

Probabilidades de la regresión logística para cada ejemplo. El último contrato está registrado
como 72 (construcción) pero su texto habla de mobiliario (56): es un caso de etiqueta ruidosa
que el modo *Auditar* detectaría.

In [9]:
probabilidades = modelo.predict_proba(ejemplos["lemas"].map(" ".join))
for i, fila in ejemplos.iterrows():
    orden = probabilidades[i].argsort()[::-1][:3]
    top3 = ", ".join(f"{modelo.classes_[j]} ({probabilidades[i][j]:.2f})" for j in orden)
    print(f"Registrado: {etapa_2_3['etiqueta'].iat[i]:>5} | Top 3: {top3}")

Registrado:    50 | Top 3: 50 (0.99), 90 (0.00), 47 (0.00)
Registrado:    84 | Top 3: 84 (1.00), 93 (0.00), otros (0.00)
Registrado:    15 | Top 3: 15 (0.68), 78 (0.20), 25 (0.11)
Registrado:    42 | Top 3: 42 (0.41), 70 (0.40), otros (0.04)
Registrado:    72 | Top 3: 56 (0.92), 43 (0.02), 44 (0.01)


## Etapa 9: los cuatro modelos, con el ajuste de distribución

`Predictor` aplica todo el pipeline, igual que la demo: normaliza, prepara la entrada de cada modelo y ajusta
las probabilidades a la frecuencia real de los segmentos.

Son ejemplos para ilustrar, no una evaluación: el primero y el cuarto están en los datos de entrenamiento. El de
mobiliario es de la prueba temporal (2026), así que los modelos no lo habían visto.

In [10]:
predictores = {
    "Reg. logística": Predictor.linea_base("regresion_logistica"),
    "SVM lineal": Predictor.linea_base("svm_lineal"),
    "CNN + fastText": Predictor.red_neuronal(),
    "BETO": Predictor.beto(str(carpeta_beto)),
}
for texto, etiqueta in zip(ejemplos["objeto_del_contrato"], etapa_2_3["etiqueta"]):
    print(f"{texto[:90]}... | registrado {etiqueta}")
    for nombre, predictor in predictores.items():
        top3 = ", ".join(f"{s.segmento} ({s.probabilidad:.2f})" for s in predictor.predecir(texto))
        print(f"   {nombre:<15} {top3}")

COMPRA DE PRODUCTOS DE CAFETERIA PARA LA ENTIDAD MUNICIPIO DE YUMBO VIGENCIA 2023... | registrado 50
   Reg. logística  50 (0.98), otros (0.01), 80 (0.00)
   SVM lineal      50 (0.56), 80 (0.16), otros (0.09)
   CNN + fastText  otros (0.39), 50 (0.28), 47 (0.04)


   BETO            50 (0.94), otros (0.02), 47 (0.01)
ADQUISICION DE LA POLIZA DE MANEJO GLOBAL PARA LA PERSONERIA MUNICIPAL DE SOTAQUIRA BOYACA... | registrado 84
   Reg. logística  84 (1.00), otros (0.00), 80 (0.00)
   SVM lineal      84 (0.82), otros (0.05), 80 (0.03)
   CNN + fastText  84 (1.00), 93 (0.00), 80 (0.00)
   BETO            84 (0.98), 93 (0.01), 80 (0.01)
Contratar el suministro de combustible; lubricantes; llantas y baterías para los vehículos... | registrado 15
   Reg. logística  15 (0.63), 78 (0.23), 25 (0.11)
   SVM lineal      15 (0.42), 25 (0.32), otros (0.10)
   CNN + fastText  78 (0.53), 15 (0.47), 25 (0.00)
   BETO            15 (0.76), 78 (0.17), 25 (0.03)
Adquisición de implementos; insumos y medicamentos para prestar los servicios de medicina ... | registrado 42
   Reg. logística  42 (0.40), 70 (0.25), otros (0.19)
   SVM lineal      42 (0.36), otros (0.19), 70 (0.16)
   CNN + fastText  otros (0.54), 42 (0.34), 85 (0.05)
   BETO            42 (0.62), otros (

### Modo Auditar

Se marca como sospechoso un contrato cuando el modelo está muy seguro de un segmento distinto al registrado.
Es una alerta para revisar, no una prueba de que el código esté mal.
Aquí se usa BETO; que un contrato se marque o no depende del modelo y del umbral (con la SVM, la confianza en 56
es 0,58 y no se marca).

In [11]:
auditoria = predictores["BETO"].auditar(ejemplos[["objeto_del_contrato", "codigo_de_categoria_principal"]], umbral=0.8)
auditoria[["objeto_del_contrato", "segmento_registrado", "segmento_sugerido", "probabilidad", "prob_registrado", "sospechoso"]]

,objeto_del_contrato,segmento_registrado,segmento_sugerido,probabilidad,prob_registrado,sospechoso
4,Adquisición de mobiliario para las sedes de la Delegación Departamental de la Registraduría Nacional del Estado Civil en el departamento de Magdalena.,72,56,0.937,0.006,True
1,ADQUISICION DE LA POLIZA DE MANEJO GLOBAL PARA LA PERSONERIA MUNICIPAL DE SOTAQUIRA BOYACA,84,84,0.985,0.985,False
0,COMPRA DE PRODUCTOS DE CAFETERIA PARA LA ENTIDAD MUNICIPIO DE YUMBO VIGENCIA 2023,50,50,0.938,0.938,False
2,Contratar el suministro de combustible; lubricantes; llantas y baterías para los vehículos y el aula móvil del Centro Agroturístico Regional Santander,15,15,0.764,0.764,False
3,Adquisición de implementos; insumos y medicamentos para prestar los servicios de medicina veterinaria en el municipio de Caldas; Antioquia,42,42,0.616,0.616,False
